# MQTT 實作：Subscriber（訂閱端）

<a href="https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/15.MQTT/MQTT_Demo_Subscriber.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

|[簡報](https://hackmd.io/@wiimax/ict_mqtt#/)|[Publisher](https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/15.MQTT/MQTT_Demo_Publisher.ipynb)|[Subscriber](https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/15.MQTT/MQTT_Demo_Subscriber.ipynb)|
|:-:|:-:|:-:|
|![](https://i.imgur.com/TFkKeuO.png)|![](https://i.imgur.com/uJdlGqo.png)|![](https://i.imgur.com/WVlVek0.png)|

## MQTT 實作

- 需要 MQTT 中間的 **Broker**：本範例使用公用的 `broker.emqx.io`
- 需要發布訊息的 **Publisher**：[Publisher 筆記本](https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/15.MQTT/MQTT_Demo_Publisher.ipynb)
- 需要接收訊息的 **Subscriber**：本筆記本

> 建議開兩個瀏覽器分頁，一邊執行 Publisher、一邊執行 Subscriber。先啟動 Subscriber，再啟動 Publisher，就能看到訊息即時送達。
>
> 公用 Broker 不需要帳號密碼，任何人都能訂閱同一個主題，請不要傳送個人資料、密碼或金鑰。

#### 熱門的公共 MQTT 伺服器：

![](https://i.imgur.com/GboMca8.png)

## Subscriber

訂閱 `nutc/#`，收到溫度資料就印出來；溫度達到 30 度時提醒開冷氣。不是溫度格式的訊息會被略過，不會讓程式中斷。

In [ ]:
!pip install paho-mqtt

In [ ]:
%%writefile mqtt_sub.py
# Subscriber：訂閱主題，收到溫度資料就判斷要不要開冷氣
#
# 執行：python mqtt_sub.py
# 停止：Ctrl+C（Colab 請按儲存格左側的停止鍵）

import json

import paho.mqtt.client as mqtt

BROKER = "broker.emqx.io"  # 要和 Publisher 使用同一個 Broker
PORT = 1883
TOPIC = "nutc/#"  # "#" 會收到 nutc/ 底下所有層級的主題
HOT = 30


def parse_temperature(payload):
    """從收到的 bytes 取出溫度；不是溫度格式就回傳 None。

    公共 Broker 上任何人都能發布訊息，收到的內容不一定是 JSON，
    所以先檢查格式，避免程式因為一筆怪訊息就中斷。
    """
    try:
        data = json.loads(payload.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError):
        return None

    # 接受 {"Temperature": 25, ...}，也接受只送一個數字的訊息
    value = data.get("Temperature") if isinstance(data, dict) else data
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        return None
    return value


def advice(temp):
    """依溫度回傳提醒文字；不需要提醒時回傳 None。"""
    if temp >= HOT:
        return f"溫度偵測到 {temp} 度，達到 {HOT} 度，該開冷氣啦！"
    return None


def on_connect(client, userdata, flags, reason_code, properties):
    print(f"connected with result code {reason_code}")
    if reason_code.is_failure:
        return
    # 寫在 on_connect 裡，斷線重連後也會重新訂閱
    client.subscribe(TOPIC)


def on_message(client, userdata, msg):
    print(f"{msg.topic} {msg.payload.decode('utf-8', errors='replace')}")
    temp = parse_temperature(msg.payload)
    if temp is None:
        print("  略過：不是溫度格式的訊息")
        return
    text = advice(temp)
    if text:
        print(f"  {text}")


def main():
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(BROKER, PORT, keepalive=60)
    try:
        client.loop_forever()  # 持續等待訊息
    except KeyboardInterrupt:
        print("已停止訂閱")
    finally:
        client.disconnect()


if __name__ == "__main__":
    main()

In [ ]:
!python mqtt_sub.py

程式會一直執行，要停止請按儲存格左側的停止鍵（或選單「執行階段 → 中斷執行」）。

### 練習
- 修改 Broker 為 `broker.hivemq.com`，記得 Publisher 與 Subscriber 要同步修改。
- 修改 `advice()` 的判斷條件：如果溫度 `temp` 低於 20，回傳「該蓋棉被啦」。
- 把 `TOPIC` 改成 `"nutc/csie/+/sensor01"`，比較萬用字元 `+`（只比對一層）與 `#`（比對底下所有層）的差別。
- 此筆記本用 `%%writefile mqtt_sub.py` 把程式存成 `.py` 檔，再用 `!python mqtt_sub.py` 在命令列執行；試著在自己電腦的終端機執行同一支腳本。

---

## Publisher（精簡版）

Colab 同一本筆記本一次只會執行一個儲存格。要在這裡執行，請先停止上方的程式；也可以把這段程式貼到另一個分頁執行。完整版請見 [Publisher 筆記本](https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/15.MQTT/MQTT_Demo_Publisher.ipynb)。

In [ ]:
# Publisher（精簡版）

import json
import time
from random import randint

import paho.mqtt.client as mqtt

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)

broker = "broker.emqx.io"
client.connect(broker, 1883, keepalive=60)

client.loop_start()

while True:
    topic = "nutc/csie/2701/sensor01"
    message = {"Temperature": randint(18, 32)}
    client.publish(topic, json.dumps(message))
    print(f"topic: {topic}, message: {message}")
    time.sleep(5)